In [4]:
#使用Keras实现ResNet-34 CNN

import keras    
class ResidualUnit(keras.layers.Layer):
    def __init__(self, filters, strides=1, activation="relu", **kwargs):
        super().__init__(**kwargs)
        self.activation = keras.activations.get(activation)
        self.main_layers = [
            keras.layers.Conv2D(filters, 3, strides=strides, padding="same", use_bias=False),
            keras.layers.BatchNormalization(),
            self.activation,
            keras.layers.Conv2D(filters, 3, strides=1, padding="same", use_bias=False),
            keras.layers.BatchNormalization()
        ]
        self.skip_layers = []
        if strides > 1:
            self.skip_layers = [
                keras.layers.Conv2D(filters, 1, strides=strides, padding="same", use_bias=False),
                keras.layers.BatchNormalization()
            ]
    
    def call(self, inputs):
        Z = inputs
        for layer in self.main_layers:
            Z = layer(Z)
        skip_Z = inputs
        for layer in self.skip_layers:
            skip_Z = layer(skip_Z)
        return self.activation(Z + skip_Z)

model = keras.models.Sequential()
model.add(keras.layers.Conv2D(64, 7, strides=2, input_shape=[224, 224, 3], padding="same", use_bias=False))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Activation("relu"))
model.add(keras.layers.MaxPool2D(pool_size=3, strides=2, padding="same"))
prev_filters = 64
for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
    strides = 1 if filters ==prev_filters else 2
    model.add(ResidualUnit(filters, strides=strides))
    prev_filters = filters
model.add(keras.layers.GlobalAvgPool2D())
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(10, activation="softmax"))

In [ ]:
#Keras 预训练
import tensorflow as tf
model = keras.applications.resnet50.ResNet50(weights="imagenet")
images_resized = tf.image.resize(images, [224, 224])
inputs = keras.applications.resnet50.preprocess_input(images_resized * 255)
Y_proba = model.predict(inputs)
top_K = keras.applications.resnet50.decode_predictions(Y_proba, top=3) 
for image_index in range(len(images)): 
    print("Image #{}".format(image_index)) 
    for class_id, name, y_proba in top_K[image_index]: 
        print("  {} - {:12s} {:.2f}%".format(class_id, name, y_proba * 100)) 
    print() 

In [ ]:
#迁移学习的预训练

import tensorflow_datasets as tfds
dataset, info = tfds.load("tf_flowers", as_supervised=True, with_info=True)
dataset_size = info.splits["train"].num_examples
class_names = info.features["label"].names
n_classes = info.features["label"].num_classes

test_split, valid_split, train_split = tfds.Split.TRAIN.subsplit([10, 15, 75]) 
test_set = tfds.load("tf_flowers", split=test_split, as_supervised=True) 
valid_set = tfds.load("tf_flowers", split=valid_split, as_supervised=True) 
train_set = tfds.load("tf_flowers", split=train_split, as_supervised=True) 

def preprocess(image, label):
    resized_image = tf.image.resize(image, [224, 224])
    final_image = keras.applications.xception.preprocess_input(resized_image)
    return final_image, label

batch_size = 32
train_set = train_set.shuffle(1000) 
train_set = train_set.map(preprocess).batch(batch_size).prefetch(1) 
valid_set = valid_set.map(preprocess).batch(batch_size).prefetch(1) 
test_set = test_set.map(preprocess).batch(batch_size).prefetch(1) 

base_model = keras.applications.xception.Xception(weights="imagenet", include_top=False) 
avg = keras.layers.GlobalAveragePooling2D()(base_model.output) 
output = keras.layers.Dense(n_classes, activation="softmax")(avg) 
model = keras.Model(inputs=base_model.input, outputs=output) 

for layer in base_model.layers:
    layer.trainable = False

optimizer = keras.optimizers.SGD(lr=0.2, momentum=0.9, decay=0.01) 
model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"]) 
history = model.fit(train_set, epochs=5, validation_data=valid_set)

for layer in base_model.layers: 
    layer.trainable = True 
optimizer = keras.optimizers.SGD(lr=0.01, momentum=0.9, decay=0.001) 
model.compile(...) 
history = model.fit(...)